# Final pipeline, end to end

**One job:** reproduce the final solution on Colab: a YOLO26l detector (`l_ps`) proposes boxes and one box-prompted U-Net (`b5_sym_s2`) segments each filament. Both trained models are in `weights/`, so nothing is retrained by default.

**Data:** put the competition download at `/content/kaggle_data`, or leave it to the next cell, which needs a Kaggle token at `~/.kaggle/access_token` from an account that joined the competition. The images are not in this repo because they cannot be redistributed.

**Steps:** setup, optional training, load the models, demo, validation PQ, test submission. With the saved weights the notebook runs on a T4 in about 10 minutes: the demo takes under a minute, validation ~3 minutes and test ~5 minutes. Set `RUN_VALIDATION` or `RUN_TEST` to `False` to skip those, or `TRAIN_DETECTOR` / `TRAIN_UNET` to `True` to retrain from scratch on an A100 (see Training).

In [ ]:
!git clone -q https://github.com/franciscomendez114/solar-filament-segmentation.git /content/solar-filament-segmentation
%cd /content/solar-filament-segmentation
%pip install -q ultralytics==8.4.191 pycocotools segmentation-models-pytorch==0.5.0 kaggle==2.2.4
!test -d /content/kaggle_data || (kaggle competitions download -c filament-segmentation-2026 -p /content && unzip -q /content/filament-segmentation-2026.zip -d /content/kaggle_data)

In [ ]:
import json
import shutil
import sys
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from pycocotools import mask as mask_utils
from ultralytics import YOLO
from ultralytics.utils import LOGGER

LOGGER.setLevel("ERROR")  # two deprecation warnings per image otherwise
sys.path.insert(0, "/content/solar-filament-segmentation")
from src.config import DEFAULT_PIPELINE_SETTINGS
from src.data import load_records, verify_or_write_split
from src.inference import cache_proposals, create_submission, fuse_boxes, predict_instances
from src.metrics import aggregate_official, annotation_to_rle, official_match_counts
from src.models import RotationTTA, load_checkpoint

RUN_VALIDATION = True  # reproduces official PQ 0.4666 on the frozen 106-image split
RUN_TEST = True  # writes submission.csv for the 180 test images
TRAIN_DETECTOR = False  # True retrains l_ps from COCO weights (A100, ~2 h); False loads weights/l_ps.pt
TRAIN_UNET = False  # True retrains b5_sym_s2 from ImageNet weights (A100); False loads weights/b5_sym_s2.pt
DETECTOR_WEIGHTS, UNET_WEIGHTS = "weights/l_ps.pt", "weights/b5_sym_s2.pt"
DETECTOR_CONFIDENCE, MASK_THRESHOLD = 0.25, 0.45
POOL_URL = "https://github.com/franciscomendez114/solar-filament-segmentation/releases/download/gong-pool-v1/gong_pool.tar"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_dir = next(Path("/content/kaggle_data").rglob("train_images")).parent
test_paths = sorted(next(Path("/content/kaggle_data").rglob("test_images")).glob("*.jpeg"))
_, records = load_records(train_dir / "MAGFiLO_1.0_Annotations_kaggle2026_train.json", train_dir / "train_images")
_, _, validation_records = verify_or_write_split(records, Path("data/processed/split_manifest.json"))
print(device, len(validation_records), "validation records,", len(test_paths), "test images")

## Training (optional)

Off by default: the notebook then uses the saved weights. With a flag set, the cell below runs the job script that trained the submitted model, with the exact settings in `configs/`, and the rest of the notebook uses the new weights. A retrained model will be close to the saved one but not identical, since GPU training is not deterministic.

**Detector** (`TRAIN_DETECTOR`, `scripts/detector_train_job.py`, about 2 hours on an A100; batch 8 at 1280 px needs ~24 GB, more than a T4 or L4 has):
1. Converts every annotator record's polygons to YOLO boxes on the frozen split (`data/processed/split_manifest.json`).
2. Adds 1,000 unlabeled GONG frames, downloaded from this repo's release, with the pseudo-labels in `data/processed/pseudo_boxes.json`. Our 24-view teacher ensemble drew them at fused score >= 0.30; the teachers are not in the repo, their labels are.
3. Fine-tunes YOLO26l from COCO weights at 1280 px: batch 8, mosaic 1.0, horizontal and vertical flips, stopping after 25 epochs without a validation mAP gain.

**U-Net** (`TRAIN_UNET`, `scripts/unet_train_job.py`, A100 recommended):
1. Cuts one 512 px crop per annotated filament (grayscale, CLAHE, box prompt) with random expansion 1.25–1.75 and center jitter, plus hard-negative crops (`data/processed/hard_negatives/`).
2. Trains an EfficientNet-B5 U-Net with scSE attention from ImageNet weights on BCE + symmetric focal-Tversky loss, seed 2.
3. After each epoch, runs the whole pipeline on validation with cached detector boxes (`data/processed/validation_proposals_yolo26x.json`), keeps the epoch with the best official PQ, and stops after 4 epochs without a gain.

In [ ]:
work = Path("/content")  # the job scripts read their inputs here, as on the training VMs


def run_job(script):
    (work / "DONE.json").unlink(missing_ok=True)
    get_ipython().system(f"python {script}")
    status = json.loads((work / "DONE.json").read_text())
    assert status["status"] == "ok", status.get("error")


if TRAIN_DETECTOR or TRAIN_UNET:
    if not (work / "src").exists():
        (work / "src").symlink_to(Path.cwd() / "src")
    shutil.copy2("data/processed/split_manifest.json", work / "split_manifest.json")
if TRAIN_DETECTOR:
    shutil.copy2("configs/final_detector.json", work / "detector_runs.json")
    shutil.copy2("data/processed/pseudo_boxes.json", work / "pseudo_boxes.json")
    if not (work / "gong_pool.tar").exists():
        get_ipython().system(f"wget -q -O /content/gong_pool.tar {POOL_URL}")
    run_job("scripts/detector_train_job.py")
    (work / "gong_pool.tar").unlink()  # the U-Net job would otherwise switch to its pseudo-mask mode
    DETECTOR_WEIGHTS = "/content/runs/l_ps/weights/best.pt"
if TRAIN_UNET:
    shutil.copy2("configs/final_unet.json", work / "unet_runs.json")
    shutil.copy2("data/processed/hard_negatives/hard_negatives_yolo26x_conf005.json", work / "hard_negatives_yolo26x_conf005.json")
    shutil.copy2("data/processed/validation_proposals_yolo26x.json", work / "validation_proposals_yolo26x.json")
    run_job("scripts/unet_train_job.py")
    UNET_WEIGHTS = "/content/results/b5_sym_s2.pt"
print("detector:", DETECTOR_WEIGHTS, "| U-Net:", UNET_WEIGHTS)

## Load the models

The saved weights, or the ones trained above. Stage 1 keeps boxes with confidence >= 0.25; Stage 2 averages each crop's mask over 8 rotations and flips.

In [ ]:
detector = YOLO(DETECTOR_WEIGHTS)
unet = RotationTTA(load_checkpoint(UNET_WEIGHTS, "b5", device)[0]).eval()  # 8-way rotation and flip TTA
settings = {**DEFAULT_PIPELINE_SETTINGS, "yolo_confidence": 0.01, "yolo_image_size": 1280}


def final_boxes(image_paths):
    # Stage 1: confidence >= 0.25, after the fusion the ensemble used, which with one detector only merges near-identical boxes
    proposals = {}
    for stem, candidates in cache_proposals(detector, image_paths, device, settings).items():
        boxes, scores = fuse_boxes([candidates], [1.0], 0.40, 0.01)
        keep = scores >= DETECTOR_CONFIDENCE
        proposals[stem] = (np.asarray(boxes[keep], dtype=np.float32).reshape(-1, 4), np.asarray(scores[keep], dtype=np.float32))
    return proposals

## Demo: one validation image

Left: the detector's boxes. Right: the final non-overlapping masks, drawn dilated so thin filaments stay visible.

In [ ]:
import cv2

record = validation_records[0]
path = record["image_path"]
boxes, scores = final_boxes([path])[Path(path).stem]
masks, _ = predict_instances(path, None, unet, device, MASK_THRESHOLD, DEFAULT_PIPELINE_SETTINGS, proposals=(boxes, scores))
image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
overlay = np.zeros((*image.shape, 4))
for mask in masks:
    overlay[cv2.dilate(mask.astype(np.uint8), np.ones((7, 7), np.uint8)) > 0] = (0.1, 0.9, 0.4, 0.9)
figure, axes = plt.subplots(1, 2, figsize=(12, 6))
for axis in axes:
    axis.imshow(image, cmap="gray", vmin=0, vmax=255)
    axis.axis("off")
for x1, y1, x2, y2 in boxes:
    axes[0].add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="orange", linewidth=1))
axes[0].set_title(f"{len(boxes)} detector boxes")
axes[1].imshow(overlay)
axes[1].set_title(f"{len(masks)} final masks")
plt.show()

## Validation: official PQ

Scores each annotator record separately and pools the counts, as the organizers' code does. Expected: PQ 0.4666, TP 828, FP 360, FN 430.

In [ ]:
if RUN_VALIDATION:
    by_image = defaultdict(list)
    for record in validation_records:
        by_image[record["physical_id"]].append(record)
    proposals = final_boxes([group[0]["image_path"] for group in by_image.values()])
    rows = []
    for physical_id, group in by_image.items():
        masks, _ = predict_instances(group[0]["image_path"], None, unet, device, MASK_THRESHOLD, DEFAULT_PIPELINE_SETTINGS, proposals=proposals[physical_id])
        predicted = [mask_utils.encode(np.asfortranarray(mask)) for mask in masks]
        for record in group:
            targets = [annotation_to_rle(annotation, record["height"], record["width"]) for annotation in record["annotations"]]
            rows.append({"physical_id": physical_id, "annotator_image_id": record["annotator_image_id"], "predicted_instances": len(masks), **official_match_counts(predicted, targets)})
    print(aggregate_official(pd.DataFrame(rows)))

## Test: Kaggle submission

Writes `submission.csv` after checking for duplicate ids and overlapping masks. This pipeline scored 0.39 on the public leaderboard.

In [ ]:
if RUN_TEST:
    submission, _ = create_submission(test_paths, None, unet, device, MASK_THRESHOLD, DEFAULT_PIPELINE_SETTINGS, final_boxes(test_paths))
    submission.to_csv("submission.csv", index=False)
    print(len(submission), "masks in submission.csv")

## Where the training inputs came from

- **Baseline detectors and U-Net:** notebooks `02` and `03`, and `scripts/detector_train_job.py` for the YOLO11m member. With their mirrored views they are the 24 teacher views (`configs/teachers.json`).
- **Unlabeled pool:** `scripts/gong_pool.py` picked and downloaded the 1,000 GONG frames (`data/processed/gong_pool.json`), and `scripts/check_gong_pool.py` shows none is a test image.
- **Pseudo-labels:** `scripts/detector_train_job.py` fused the teacher views at score >= 0.30 into `data/processed/pseudo_boxes.json`.
- **Hard negatives and validation boxes for U-Net selection:** cached from the September YOLO26x detector, by notebook `02` (`MINE_HARD_NEGATIVES`) and `scripts/experiments/build_validation_proposals.py`.